# Chequeo visual rápido: SpiderCamModel + cinemática de cables

Antes de tener la visualización final en PyQtGraph (Fase 9), este notebook
sirve para verificar "a ojo" con matplotlib que el carro se mueve dentro
del volumen de trabajo y que las longitudes de cable tienen sentido.

Requiere haber corrido `pip install -e .` en la raíz del proyecto.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from spidercam.simulation.stadium import Stadium
from spidercam.simulation.spidercam_model import SpiderCamModel

stadium = Stadium(
    field_length_m=105,
    field_width_m=68,
    anchor_points=[(0, 0, 25), (105, 0, 25), (105, 68, 25), (0, 68, 25)],
)
model = SpiderCamModel(stadium, initial_position=[52.5, 34, 15], max_speed_m_s=5.0)
print("Volumen de trabajo:", stadium.workspace_bounds)

## Simular una trayectoria circular y registrar posición + longitudes de cable

In [ ]:
dt = 0.1
steps = 200
positions = []
cable_lengths_history = []

for i in range(steps):
    t = i * dt
    # velocidad tangencial a un círculo de radio 20 m alrededor del centro de la cancha
    vx = -20 * np.sin(t)
    vy = 20 * np.cos(t)
    model.update(velocity_command=[vx, vy, 0.0], dt=dt)
    positions.append(model.position.copy())
    cable_lengths_history.append(model.get_cable_lengths())

positions = np.array(positions)
cable_lengths_history = np.array(cable_lengths_history)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(positions[:, 0], positions[:, 1])
axes[0].set_title("Trayectoria del carro (vista superior)")
axes[0].set_xlabel("X (m)")
axes[0].set_ylabel("Y (m)")
axes[0].set_xlim(0, stadium.field_length_m)
axes[0].set_ylim(0, stadium.field_width_m)
axes[0].set_aspect("equal")

for cable_idx in range(4):
    axes[1].plot(cable_lengths_history[:, cable_idx], label=f"cable {cable_idx + 1}")
axes[1].set_title("Longitud de cada cable en el tiempo")
axes[1].set_xlabel("paso de simulación")
axes[1].set_ylabel("longitud (m)")
axes[1].legend()

plt.tight_layout()
plt.show()